# Wikipedia RAG Indexing Pipeline (Elasticsearch)

Creates an Elasticsearch index for RAG evaluation with popularity metadata.

## Advantages over Chroma version:
- Single unified index for vector + BM25 search
- Production-grade scalability
- Native hybrid search support
- Can switch strategies at query time

## Prerequisites:
```bash
# Install Elasticsearch integration
pip install -qU langchain-elasticsearch

# Start Elasticsearch locally (choose one):
# Option 1 - Quick start (recommended):
curl -fsSL https://elastic.co/start-local | sh

# Option 2 - Manual Docker:
docker run -d --name elasticsearch \
    -p 9200:9200 -p 9300:9300 \
    -e "discovery.type=single-node" \
    -e "xpack.security.enabled=false" \
    docker.elastic.co/elasticsearch/elasticsearch:8.12.0
```

## Steps
1. Load QA datasets from HuggingFace
2. Load Wikipedia corpus
3. Add popularity metadata
4. Create Elasticsearch index (supports both vector and BM25)

In [3]:
from pathlib import Path
import os
import sys
import pandas as pd
from datasets import load_dataset, concatenate_datasets
from rag.elasticsearch_rag_service import ElasticsearchRagService
from rag.utils import IndexingConfig
from config import DATA_DIR, CACHE_DIR
from llm.openAi_service import OpenAIService
import numpy as np
from tqdm.auto import tqdm
import asyncio
import logging
import dotenv

dotenv.load_dotenv()

# Suppress noisy HTTP logs from libraries
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)
logging.getLogger("elastic_transport").setLevel(logging.WARNING)

# ============================================================================
# CONFIGURATION
# ============================================================================

# Elasticsearch Connection
ES_URL = "http://localhost:9200"
ES_USER = os.getenv("ELASTICSEARCH_USERNAME")  # Set if using authentication
ES_PASSWORD = os.getenv("ELASTICSEARCH_PASSWORD")  # Set if using authentication

# Datasets
QA_DATASETS = ['natural_questions']
WIKIPEDIA_DATASET = "facebook/kilt_wikipedia"
WIKIPEDIA_VERSION = "2019-08-01"
POPULARITY_DATASET = "Cyro1/enwiki_pageviews_m"

# Indexing
NAME = "wiki_150k_balanced_qa_nq"
COLLECTION_NAME = NAME
N_RANDOM_SAMPLES = 150_000  # Set to None to use all data
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"
STRATEGY = "hybrid"  # "vector", "bm25", or "hybrid" (recommended)

# Paths - Updated to separate index and questions
COLLECTION_ROOT = Path(DATA_DIR) / NAME
COLLECTION_PATH = COLLECTION_ROOT / "index"  # For metadata (ES index is in Elasticsearch)
QUESTIONS_PATH = COLLECTION_ROOT / "train_questions.parquet"

# Processing
BATCH_SIZE = 1000
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# Balancing
BALANCE_DECILES = True  # Ensure equal questions across 10 popularity deciles
ADD_SYNTHETIC_QUESTIONS = False  # Generate questions for under-represented deciles
MIN_QUESTIONS_PER_DECILE = 200  # Target number of questions per decile
MODEL_NAME = "gpt-4.1-nano"  # Model for synthetic question generation

# Async generation settings
SYNTHETIC_BATCH_SIZE = 500  # How many questions to generate in parallel

print(f"✓ Config loaded: {QA_DATASETS} → {COLLECTION_NAME} (Elasticsearch)")
print(f"  Strategy: {STRATEGY}")
print(f"  ES URL: {ES_URL}")
print(f"  Output Root: {COLLECTION_ROOT}")
print(f"  Balance: {BALANCE_DECILES} (Synthetic: {ADD_SYNTHETIC_QUESTIONS}, Target: {MIN_QUESTIONS_PER_DECILE})")

✓ Config loaded: ['natural_questions'] → wiki_150k_balanced_qa_nq (Elasticsearch)
  Strategy: hybrid
  ES URL: http://localhost:9200
  Output Root: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_150k_balanced_qa_nq
  Balance: True (Synthetic: False, Target: 200)


In [4]:
# ============================================================================
# STEP 1: Load QA Datasets & Balance Early
# ============================================================================

print("Loading QA datasets...")
qa_dfs = []

if not QA_DATASETS:
    print("  No QA datasets specified. Proceeding without QA data.")
    qa_df = pd.DataFrame(columns=[
        "question_text",
        "answers",
        "wikipedia_id",
        "dataset",
        "popularity_avg",
        "popularity_rank",
        "decile",
        "is_synthetic",
        "wikipedia_title",
    ])
    required_doc_ids = set()
else:
    for ds_name in QA_DATASETS:
        ds = load_dataset("Cyro1/popularity-enriched-qa-datasets", ds_name, split="train+test", cache_dir=CACHE_DIR)
        df = ds.to_pandas()
        df["dataset"] = ds_name
        qa_dfs.append(df)
        print(f"  {ds_name}: {len(df):,} questions")

    qa_df = pd.concat(qa_dfs, ignore_index=True)

    # Normalize rank column naming
    if "rank_avg" in qa_df.columns:
        if "popularity_rank" in qa_df.columns:
            qa_df["popularity_rank"] = qa_df["popularity_rank"].combine_first(qa_df["rank_avg"])
            qa_df = qa_df.drop(columns=["rank_avg"])
        else:
            qa_df = qa_df.rename(columns={"rank_avg": "popularity_rank"})

    # Clean IDs
    qa_df = qa_df.dropna(subset=["wikipedia_id"])
    qa_df["wikipedia_id"] = qa_df["wikipedia_id"].astype(int)

    print(f"\n✓ Initial QA Set: {len(qa_df):,} questions")

    # ============================================================================
    # EARLY BALANCING - Calculate deciles first if needed
    # ============================================================================

    if BALANCE_DECILES and not ADD_SYNTHETIC_QUESTIONS:
        print("\n⚖️ Early Balancing (calculating deciles from popularity data)...")
        
        # Load popularity dataset to calculate deciles
        print("  Loading popularity data for decile calculation...")
        pop_ds = load_dataset(POPULARITY_DATASET, split="train+test", cache_dir=CACHE_DIR)
        
        # Detect columns
        cols = pop_ds.column_names
        id_col = "wikipedia_id" if "wikipedia_id" in cols else "id"
        
        # Load and calculate global deciles
        print("  Calculating global deciles...")
        pop_df_minimal = pop_ds.select_columns([id_col, "popularity_avg"]).to_pandas()
        pop_df_minimal[id_col] = pd.to_numeric(pop_df_minimal[id_col], errors='coerce').fillna(-1).astype(int)
        
        # Calculate deciles on full dataset
        pop_df_minimal["decile"] = pd.qcut(
            pop_df_minimal["popularity_avg"].rank(method="first"),
            10,
            labels=False,
        )
        
        # Create lookup for QA dataset
        decile_lookup = pop_df_minimal.set_index(id_col)["decile"].to_dict()
        
        # Add deciles to QA dataset
        qa_df["decile"] = qa_df["wikipedia_id"].map(decile_lookup)
        qa_df["decile"] = qa_df["decile"].fillna(-1).astype(int)
        
        # Clean up
        del pop_ds, pop_df_minimal
        import gc
        gc.collect()
        
        # Drop unknown decile questions
        invalid = (qa_df["decile"] == -1).sum()
        if invalid:
            print(f"  Dropped {invalid} questions with unknown decile")
        qa_df = qa_df[qa_df["decile"] != -1].copy()
        
        counts = qa_df["decile"].value_counts().sort_index()
        print(f"  Current Distribution:\n{counts}")
        
        # Balance to smallest decile
        target_count = counts.min()
        print(f"  Target per decile: {target_count}")
        
        balanced_dfs = []
        for decile in range(10):
            decile_df = qa_df[qa_df["decile"] == decile]
            if len(decile_df) > target_count:
                decile_df = decile_df.sample(n=target_count, random_state=42)
            balanced_dfs.append(decile_df)
        
        qa_df = pd.concat(balanced_dfs, ignore_index=True)
        print(f"  ✓ Balanced to {len(qa_df):,} questions")
        print(f"  New Distribution:\n{qa_df['decile'].value_counts().sort_index()}")
        
        required_doc_ids = set(qa_df["wikipedia_id"])
    else:
        required_doc_ids = set(qa_df["wikipedia_id"])

print(f"\n✓ Documents needed: {len(required_doc_ids):,}")


Loading QA datasets...
  natural_questions: 81,533 questions

✓ Initial QA Set: 81,533 questions

⚖️ Early Balancing (calculating deciles from popularity data)...
  Loading popularity data for decile calculation...
  Calculating global deciles...
  Current Distribution:
decile
0       28
1       10
2       25
3       43
4       90
5      148
6      383
7     1019
8     3837
9    75950
Name: count, dtype: int64
  Target per decile: 10
  ✓ Balanced to 100 questions
  New Distribution:
decile
0    10
1    10
2    10
3    10
4    10
5    10
6    10
7    10
8    10
9    10
Name: count, dtype: int64

✓ Documents needed: 95


In [6]:
# ============================================================================
# STEP 2: Load Wikipedia (Only Required Documents)
# ============================================================================

print("\nLoading Wikipedia...")
# Keep a reference to full dataset for retrieval
full_wiki_ds = load_dataset(WIKIPEDIA_DATASET, WIKIPEDIA_VERSION, split="full", cache_dir=CACHE_DIR)
full_wiki_ds = full_wiki_ds.select_columns(["wikipedia_id", "wikipedia_title", "text"])

if N_RANDOM_SAMPLES is not None and len(required_doc_ids) < N_RANDOM_SAMPLES:
    print(f"Sampling {N_RANDOM_SAMPLES:,} random documents from Wikipedia...")
    wiki_ds = full_wiki_ds.shuffle(seed=42).select(range(N_RANDOM_SAMPLES))
    print(f"✓ Loaded {len(wiki_ds):,} articles")
    
    # Check for missing required documents
    print("Checking for missing required documents...")
    existing_ids = set(int(i) for i in wiki_ds["wikipedia_id"] if i is not None)
    missing_ids = required_doc_ids - existing_ids
    print(f"✓ Missing: {len(missing_ids):,}")
    
    # Add missing docs if needed (OPTIMIZED - batched filtering)
    if missing_ids:
        print("  Fetching missing documents (this may take a moment)...")
        missing_ds = full_wiki_ds.filter(
            lambda batch: [int(i) in missing_ids for i in batch["wikipedia_id"]],
            batched=True,
            batch_size=10000,
            desc="Finding missing docs"
        )
        wiki_ds = concatenate_datasets([wiki_ds, missing_ds])
        print(f"✓ Added {len(missing_ds):,} missing docs. Total: {len(wiki_ds):,}")
else:
    # Load only required documents (OPTIMIZED - batched filtering)
    print(f"Loading only {len(required_doc_ids):,} required documents...")
    if required_doc_ids:
        wiki_ds = full_wiki_ds.filter(
            lambda batch: [int(i) in required_doc_ids for i in batch["wikipedia_id"]],
            batched=True,
            batch_size=10000,
            desc="Loading required docs"
        )
        print(f"✓ Loaded {len(wiki_ds):,} articles")
    else:
        print("⚠️  No documents required, using empty dataset")
        wiki_ds = full_wiki_ds.select([])

# Flatten KILT text structure - ALWAYS CHECK
print("Normalizing text format...")

def flatten_text(batch):
    return {
        "text": [
            "\n".join(t["paragraph"]) if isinstance(t, dict) and "paragraph" in t else str(t)
            for t in batch["text"]
        ]
    }

# Apply to first item to check if needed
needs_flattening = False
if len(wiki_ds) > 0:
    sample_text = wiki_ds[0]["text"]
    if isinstance(sample_text, dict):
        needs_flattening = True

if needs_flattening:
    wiki_ds = wiki_ds.map(flatten_text, batched=True, desc="Flattening text")
    print("✓ Text flattened")
else:
    print("✓ Text already flat (or empty)")



Loading Wikipedia...


Loading dataset shards:   0%|          | 0/59 [00:00<?, ?it/s]

Sampling 150,000 random documents from Wikipedia...
✓ Loaded 150,000 articles
Checking for missing required documents...
✓ Missing: 91
  Fetching missing documents (this may take a moment)...


Finding missing docs:   0%|          | 0/5903530 [00:00<?, ? examples/s]

✓ Added 91 missing docs. Total: 150,091
Normalizing text format...


Flattening text:   0%|          | 0/150091 [00:00<?, ? examples/s]

✓ Text flattened


In [7]:
# ============================================================================
# STEP 3: Add Popularity Metadata
# ============================================================================

print("Loading popularity data...")
pop_ds = load_dataset(POPULARITY_DATASET, split="train+test", cache_dir=CACHE_DIR)

# Detect columns
cols = pop_ds.column_names
id_col = "wikipedia_id" if "wikipedia_id" in cols else "id"
rank_col = next((c for c in ["rank_avg", "avg_rank"] if c in cols), None)

# Get needed IDs early
needed_ids = set(int(x) for x in wiki_ds["wikipedia_id"] if x is not None)
print(f"✓ Target documents: {len(needed_ids):,}")

print("Processing popularity metadata (OPTIMIZED - Vectorized)...")

# STEP 1: Load MINIMAL data first to calculate global deciles
print("  Loading popularity scores for global decile calculation...")
pop_df_minimal = pop_ds.select_columns([id_col, "popularity_avg"]).to_pandas()
pop_df_minimal[id_col] = pd.to_numeric(pop_df_minimal[id_col], errors='coerce').fillna(-1).astype(int)

# STEP 2: Calculate Global Deciles (Vectorized on full dataset)
print("  Calculating global deciles...")
pop_df_minimal["decile"] = pd.qcut(
    pop_df_minimal["popularity_avg"].rank(method="first"),
    10,
    labels=False,
)

# STEP 3: FILTER EARLY - Keep only needed rows
print(f"  Filtering from {len(pop_df_minimal):,} to {len(needed_ids):,} rows...")
relevant_pop_df = pop_df_minimal[pop_df_minimal[id_col].isin(needed_ids)].copy()

# STEP 4: Add rank column if needed
if rank_col and rank_col in pop_ds.column_names:
    print("  Adding rank data for relevant documents only...")
    pop_ds_filtered = pop_ds.filter(
        lambda batch: [int(i) in needed_ids for i in batch[id_col]],
        batched=True,
        batch_size=10000,
        desc="Filtering popularity data",
    )
    rank_df = pop_ds_filtered.select_columns([id_col, rank_col]).to_pandas()
    rank_df[id_col] = rank_df[id_col].astype(int)
    relevant_pop_df = relevant_pop_df.merge(rank_df, on=id_col, how="left")

# Standardize rank column name
if rank_col:
    relevant_pop_df = relevant_pop_df.rename(columns={rank_col: "popularity_rank"})
else:
    relevant_pop_df["popularity_rank"] = None

# STEP 5: Build lookup dictionary
print("  Building optimized lookup...")
pop_lookup = relevant_pop_df.set_index(id_col).to_dict(orient="index")

# Cleanup
del pop_df_minimal, pop_ds
if rank_col:
    del pop_ds_filtered, rank_df
import gc
gc.collect()

print(f"✓ Built lookup with {len(pop_lookup):,} entries")

# Merge with Wikipedia using batched map
print("Merging metadata...")

def merge_batch(batch):
    """Vectorized metadata merge"""
    ids = [int(i) for i in batch["wikipedia_id"]]
    defaults = {"popularity_avg": None, "popularity_rank": None, "decile": -1}
    meta_list = [pop_lookup.get(i, defaults) for i in ids]

    return {
        "popularity_avg": [m.get("popularity_avg") for m in meta_list],
        "popularity_rank": [m.get("popularity_rank") for m in meta_list],
        "decile": [m.get("decile", -1) for m in meta_list],
    }

wiki_ds_with_pop = wiki_ds.map(
    merge_batch,
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Merging",
)

print(f"✓ Ready for indexing: {len(wiki_ds_with_pop):,} documents")

Loading popularity data...
✓ Target documents: 150,091
Processing popularity metadata (OPTIMIZED - Vectorized)...
  Loading popularity scores for global decile calculation...
  Calculating global deciles...
  Filtering from 5,903,530 to 150,091 rows...
  Adding rank data for relevant documents only...


Filtering popularity data:   0%|          | 0/5903530 [00:00<?, ? examples/s]

  Building optimized lookup...
✓ Built lookup with 150,091 entries
Merging metadata...


Merging:   0%|          | 0/150091 [00:00<?, ? examples/s]

✓ Ready for indexing: 150,091 documents


In [8]:
# ============================================================================
# STEP 3.5: Load Synthetic Question Generation Prompt
# ============================================================================

if ADD_SYNTHETIC_QUESTIONS:
    PROMPT_FILE = Path(DATA_DIR) / "prompts" / "synthentic_question_generation_promt.txt"

    print(f"Loading prompt template from {PROMPT_FILE}...")
    if PROMPT_FILE.exists():
        QUESTION_GENERATION_PROMPT = PROMPT_FILE.read_text().strip()
        print("✓ Loaded prompt template:")
        print(f"  {QUESTION_GENERATION_PROMPT[:100]}...")
    else:
        print(f"⚠️  Prompt file not found at {PROMPT_FILE}")
        print("  Using default prompt...")
        QUESTION_GENERATION_PROMPT = "Given this context/document passage, generate one or more relevant questions that a user might ask based on the passage.\n\nDocument: {passage}"

    print()

In [9]:
# ============================================================================
# STEP 3.6: Final Balancing & Synthetic Questions (if needed)
# ============================================================================

CREATE_DATA = False  # Set to True to load existing data instead

if CREATE_DATA:
    print("Loading existing questions...")
    qa_df = pd.read_parquet(QUESTIONS_PATH)
elif ADD_SYNTHETIC_QUESTIONS:
    print("\n⚖️ Generating Synthetic Questions...")

    # Map questions to deciles + popularity
    decile_map = {doc_id: meta.get("decile", -1) for doc_id, meta in pop_lookup.items()}
    pop_map = {doc_id: meta.get("popularity_avg") for doc_id, meta in pop_lookup.items()}
    rank_map = {doc_id: meta.get("popularity_rank") for doc_id, meta in pop_lookup.items()}

    qa_df["decile"] = qa_df["wikipedia_id"].map(decile_map)

    if "popularity_avg" in qa_df.columns:
        qa_df["popularity_avg"] = qa_df["popularity_avg"].combine_first(qa_df["wikipedia_id"].map(pop_map))
    else:
        qa_df["popularity_avg"] = qa_df["wikipedia_id"].map(pop_map)

    if "popularity_rank" in qa_df.columns:
        qa_df["popularity_rank"] = qa_df["popularity_rank"].combine_first(qa_df["wikipedia_id"].map(rank_map))
    else:
        qa_df["popularity_rank"] = qa_df["wikipedia_id"].map(rank_map)

    qa_df["decile"] = qa_df["decile"].fillna(-1).astype(int)
    qa_df["is_synthetic"] = False

    # Drop unknown decile questions
    invalid = (qa_df["decile"] == -1).sum()
    if invalid:
        print(f"  Warning: Dropped {invalid} questions with unknown decile")
    qa_df = qa_df[qa_df["decile"] != -1].copy()

    counts = qa_df["decile"].value_counts().sort_index()
    print(f"  Current Distribution:\n{counts}")
    print(f"  Target per decile: {MIN_QUESTIONS_PER_DECILE}")

    # Init LLM
    try:
        llm_service = OpenAIService(temperature=0.7, request_timeout=None, model_name=MODEL_NAME)
        print("  ✓ LLM Service Initialized")
    except Exception as e:
        print(f"  ❌ Failed to init LLM: {e}")
        print("  Disabling synthetic generation.")
        llm_service = None

    if llm_service:
        # Precompute decile indices
        wiki_deciles = np.asarray(wiki_ds_with_pop["decile"])
        decile_to_indices = {decile: np.where(wiki_deciles == decile)[0] for decile in range(10)}

        async def _generate_questions_for_decile(decile: int, needed: int):
            indices = decile_to_indices.get(decile)
            if indices is None or len(indices) == 0:
                return []

            idxs = np.random.choice(indices, size=needed, replace=(needed > len(indices)))
            batch_size = min(SYNTHETIC_BATCH_SIZE, max(1, needed))

            async def _gen_one(idx):
                doc = wiki_ds_with_pop[int(idx)]
                text = doc["text"]
                doc_id = int(doc["wikipedia_id"])

                prompt = QUESTION_GENERATION_PROMPT.format(passage=f"{text[:2000]}")

                try:
                    response = await llm_service.ainvoke(prompt)
                    question = response.strip()
                    if not question:
                        return None
                    return {
                        "question_text": question,
                        "answer_texts": [],
                        "wikipedia_id": doc_id,
                        "wikipedia_title": doc.get("wikipedia_title"),
                        "dataset": "synthetic",
                        "decile": decile,
                        "is_synthetic": True,
                        "popularity_avg": doc.get("popularity_avg"),
                        "popularity_rank": doc.get("popularity_rank"),
                    }
                except Exception:
                    return None

            tasks = [_gen_one(i) for i in idxs]
            results = []
            for start in range(0, len(tasks), batch_size):
                batch = tasks[start : start + batch_size]
                for coro in tqdm(asyncio.as_completed(batch), total=len(batch), desc=f"Decile {decile}"):
                    result = await coro
                    if result:
                        results.append(result)
            return results

        def _run_async(coro):
            try:
                loop = asyncio.get_running_loop()
            except RuntimeError:
                return asyncio.run(coro)
            else:
                import nest_asyncio
                nest_asyncio.apply()
                return loop.run_until_complete(coro)

        final_dfs = []
        for decile in range(10):
            current_df = qa_df[qa_df["decile"] == decile]
            curr_count = len(current_df)

            if curr_count >= MIN_QUESTIONS_PER_DECILE:
                final_dfs.append(current_df.sample(n=MIN_QUESTIONS_PER_DECILE, random_state=42))
                continue

            final_dfs.append(current_df)

            needed = MIN_QUESTIONS_PER_DECILE - curr_count
            print(f"  Decile {decile}: Generating {needed} synthetic questions...")

            new_qs = _run_async(_generate_questions_for_decile(decile, needed))

            if new_qs:
                final_dfs.append(pd.DataFrame(new_qs))
                print(f"    ✓ Generated {len(new_qs)} questions")
            if len(new_qs) < needed:
                print(f"    ⚠️ Shortfall: {needed - len(new_qs)} questions")

        qa_df = pd.concat(final_dfs, ignore_index=True)

    # Ensure consistent schema and types
    if "question_text" not in qa_df.columns and "question" in qa_df.columns:
        qa_df = qa_df.rename(columns={"question": "question_text"})

    qa_df["wikipedia_id"] = qa_df["wikipedia_id"].astype(int)

    print(f"✓ Final Total: {len(qa_df):,} questions")
    print(f"Final Distribution:\n{qa_df['decile'].value_counts().sort_index()}")
elif BALANCE_DECILES and "decile" in qa_df.columns:
    print(f"\n✓ Already balanced: {len(qa_df):,} questions")
else:
    print(f"\n✓ Using unbalanced dataset: {len(qa_df):,} questions")

display(qa_df.head())



✓ Already balanced: 100 questions


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank,dataset,decile
0,4339184963143653248,what does tfsi stand for on an audi tt,"[Turbocharged fuel stratified injection, ]",23725531,Turbo fuel stratified injection,7.172348,5.314850e+06,natural_questions,0
1,279594629442700387,who was the first ruler in the recorded histor...,"[King Devanampiya Tissa, ]",684137,Wildlife refuge,3.912500,5.550163e+06,natural_questions,0
2,-4729602067146348685,when was the last time the sahara was green,"[3900 BCE, ]",12746283,Neolithic Subpluvial,5.960227,5.362490e+06,natural_questions,0
3,2943108974768713447,who won the jeopardy college tournament of cha...,"[Tom Cubbage, ]",5575376,Jeopardy! College Championship,5.834091,5.477161e+06,natural_questions,0
4,5074106293586903195,who is the current leader of the official oppo...,"[Andrew Scheer, ]",711418,List of Leaders of the Official Opposition (Ca...,1.866667,5.801144e+06,natural_questions,0


In [10]:
# ============================================================================
# STEP 3.7: Data Sanitization for Elasticsearch
# ============================================================================

print("\n🧹 Sanitizing data for Elasticsearch indexing...")

wiki_df = wiki_ds_with_pop.to_pandas()

# Ensure numeric conversions happen after filling NaNs to avoid IntCastingNaNError
wiki_df["wikipedia_id"] = pd.to_numeric(wiki_df["wikipedia_id"], errors="coerce").astype(int)

# Use pandas nullable Float64 dtype for columns that may contain NaN
wiki_df["popularity_avg"] = pd.to_numeric(wiki_df["popularity_avg"], errors="coerce").fillna(-1).astype("Float64")
wiki_df["decile"] = pd.to_numeric(wiki_df["decile"], errors="coerce").fillna(-1).astype("Int64")
wiki_df["popularity_rank"] = pd.to_numeric(wiki_df["popularity_rank"], errors="coerce").astype("Float64")

# Show any rows that were missing popularity_avg
display(wiki_df.head())

# Save the questions used for this run
print(f"Saving training questions to {QUESTIONS_PATH}...")
qa_df.to_parquet(QUESTIONS_PATH)


🧹 Sanitizing data for Elasticsearch indexing...


,wikipedia_id,wikipedia_title,text,popularity_avg,popularity_rank,decile
0,1387498,Trade Union Coordination Centre,Trade Union Coordination Centre\n\nTrade Union...,294.895833,1505528.802083,7
1,15224061,Saye-Taayor Adolphus Dolo,Saye-Taayor Adolphus Dolo\n\nAdolphus Dolo (bo...,28.125,3900514.770833,3
2,12611627,Chiloglanis ruziziensis,Chiloglanis ruziziensis\n\nChiloglanis ruzizie...,10.645833,4904842.9375,1
3,14882009,Gankhuyagiin Oyuungerel,Gankhuyagiin Oyuungerel\n\nGankhuyagiin Oyuung...,51.791667,3194730.71875,4
4,53471602,Pınar Soykan,Pınar Soykan\n\nPınar Soykan (born 24 April 19...,34.541667,3722928.552083,3


Saving training questions to /Users/cyro/Documents/VSC/PopularityBias/data/wiki_150k_balanced_qa_nq/train_questions.parquet...


In [ ]:
# ============================================================================
# STEP 4: Create Elasticsearch Index
# ============================================================================

print("\n📊 Creating Elasticsearch index...")

# Initialize service with hybrid strategy for maximum flexibility
config = IndexingConfig(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    embedding_provider="huggingface",
    embedding_model=EMBEDDING_MODEL,
    trust_remote_code=True,
    use_progress=True
 )

service = ElasticsearchRagService(
    config=config,
    es_url=ES_URL,
    es_user=ES_USER,
    es_password=ES_PASSWORD,
    strategy=STRATEGY,
 )

# Ensure output directory exists
COLLECTION_ROOT.mkdir(parents=True, exist_ok=True)



# Use the sanitized DataFrame (wiki_df) instead of the raw dataset
print(f"Indexing {len(wiki_df):,} documents to Elasticsearch...")
index, num_chunks = service.index_from_dataframe(
    df=wiki_df,
    text_field="text",
    metadata_fields=["wikipedia_id", "wikipedia_title", "popularity_avg", "popularity_rank", "decile"],
    output_dir=COLLECTION_PATH,
    collection_name=COLLECTION_NAME,
    progress_bar=True,
    batch_size=BATCH_SIZE,
 )

print(f"\n✅ DONE")
print(f"  Documents: {len(wiki_df):,}")
print(f"  Chunks: {num_chunks:,}")
print(f"  Elasticsearch Index: {COLLECTION_NAME}")
print(f"  ES URL: {ES_URL}")
print(f"  Strategy: {STRATEGY}")
print(f"  Questions: {QUESTIONS_PATH}")
print(f"\nNext: Run rag_evaluation.ipynb")
